# 04. Tools & Structured Outputs

The model does NOT call backend services directly. Tool calling is a mechanism where the model **proposes** a structured action, and your application **validates, authorizes, and executes** it.

In this lab, we build a **defense-in-depth execution boundary** for a customer support agent assisting users with order inquiries and refunds. We explicitly test that no model proposal can bypass schema constraints, tenant isolation, business rules, or authorization gates.

In [1]:
import json
import time
import os
from typing import Optional, Literal, Set, Dict, Any
from pydantic import BaseModel, Field, ConfigDict, ValidationError
from enum import Enum

# Central model configuration
MODEL_NAME = 'gpt-4o-mini'
print(f'Environment initialized. Configured model: {MODEL_NAME}')

Environment initialized. Configured model: gpt-4o-mini


## Part 1 & 2 — Strict Argument Schemas
We define tool arguments using Pydantic models with `extra='forbid'`. This strictly prevents the model from injecting unauthorized metadata (such as overriding `tenant_id` or `actor_id`).

In [2]:
class RefundReason(str, Enum):
    DAMAGED = 'damaged'
    LOST = 'lost'
    CUSTOMER_REQUEST = 'customer_request'

class GetOrderArgs(BaseModel):
    model_config = ConfigDict(extra='forbid')
    order_id: str = Field(..., description='The alphanumeric order identifier')

class IssueRefundArgs(BaseModel):
    model_config = ConfigDict(extra='forbid')
    order_id: str = Field(..., description='The unique order identifier')
    amount_cents: int = Field(..., gt=0, description='Amount to refund in integer cents (must be > 0)')
    reason: RefundReason = Field(..., description='Approved reason category for the refund')
    idempotency_key: str = Field(..., description='Unique client idempotency token to prevent duplicate refund executions')

print('IssueRefundArgs JSON Schema:')
print(json.dumps(IssueRefundArgs.model_json_schema(), indent=2))

IssueRefundArgs JSON Schema:
{
  "$defs": {
    "RefundReason": {
      "enum": [
        "damaged",
        "lost",
        "customer_request"
      ],
      "title": "RefundReason",
      "type": "string"
    }
  },
  "additionalProperties": false,
  "properties": {
    "order_id": {
      "description": "The unique order identifier",
      "title": "Order Id",
      "type": "string"
    },
    "amount_cents": {
      "description": "Amount to refund in integer cents (must be > 0)",
      "exclusiveMinimum": 0,
      "title": "Amount Cents",
      "type": "integer"
    },
    "reason": {
      "$ref": "#/$defs/RefundReason",
      "description": "Approved reason category for the refund"
    },
    "idempotency_key": {
      "description": "Unique client idempotency token to prevent duplicate refund executions",
      "title": "Idempotency Key",
      "type": "string"
    }
  },
  "required": [
    "order_id",
    "amount_cents",
    "reason",
    "idempotency_key"
  ],
  "title": "Is

## Part 3 — Typed Result Models
Tools return strongly typed Pydantic result objects rather than unvalidated dictionaries or raw strings.

In [3]:
class OrderResult(BaseModel):
    order_id: str
    tenant_id: str
    customer_id: str
    total_cents: int
    status: str

class RefundResult(BaseModel):
    status: Literal['refund_issued', 'already_processed']
    transaction_id: str
    amount_cents: int
    order_id: str

class ErrorResult(BaseModel):
    error_type: Literal['SCHEMA_ERROR', 'BUSINESS_ERROR', 'AUTH_ERROR', 'UNKNOWN_TOOL']
    message: str
print('Result models initialized.')

Result models initialized.


## Part 4 — Actor Identity & Tenant Scope
The model never provides actor identity or permissions. The application runtime injects a trusted `ExecutionContext` representing the active support agent and organization scope.

- `actor_id`: The support agent invoking the tool (`support-agent-007`).
- `tenant_id`: The organization scope (`northstar`).
- `customer_id`: The end customer who placed the order (`CUST-101`). Orders belong to customers, not the support agent.

In [4]:
class ExecutionContext(BaseModel):
    actor_id: str = Field(..., description='Unique identifier of the agent/operator')
    tenant_id: str = Field(..., description='Tenant organization boundary')
    roles: Set[str] = Field(default_factory=set, description='Assigned authorization roles')

# Context instances for testing authorization boundaries
ctx_support_agent = ExecutionContext(
    actor_id='support-agent-007',
    tenant_id='northstar',
    roles={'order:read', 'refund:issue'}
)
ctx_read_only_agent = ExecutionContext(
    actor_id='trainee-agent-001',
    tenant_id='northstar',
    roles={'order:read'}  # Lacks refund:issue
)
ctx_other_tenant_agent = ExecutionContext(
    actor_id='external-agent-999',
    tenant_id='acme_corp',
    roles={'order:read', 'refund:issue'}
)
print('Execution contexts initialized.')

Execution contexts initialized.


## Part 5 — Tool Registry with Explicit Permissions & Idempotency
Tools are registered with explicit effect types (`read` vs `write`), required permissions, and argument schemas.

In [5]:
# Mock Database
DB_ORDERS: Dict[str, Dict[str, Any]] = {
    'ORD-123': {'tenant_id': 'northstar', 'customer_id': 'CUST-101', 'total_cents': 5000, 'status': 'delivered'},
    'ORD-456': {'tenant_id': 'northstar', 'customer_id': 'CUST-202', 'total_cents': 10000, 'status': 'delivered'},
    'ORD-789': {'tenant_id': 'other_tenant', 'customer_id': 'CUST-999', 'total_cents': 7500, 'status': 'delivered'}
}
DB_PROCESSED_REFUNDS: Set[str] = set()

# Backend Implementation Handlers
def get_order_impl(args: GetOrderArgs) -> OrderResult:
    rec = DB_ORDERS[args.order_id]
    return OrderResult(
        order_id=args.order_id,
        tenant_id=rec['tenant_id'],
        customer_id=rec['customer_id'],
        total_cents=rec['total_cents'],
        status=rec['status']
    )

def issue_refund_impl(args: IssueRefundArgs) -> RefundResult:
    # Idempotency Check: Same logical refund request returns existing result
    if args.idempotency_key in DB_PROCESSED_REFUNDS:
        return RefundResult(
            status='already_processed',
            transaction_id='tx_existing_prev',
            amount_cents=args.amount_cents,
            order_id=args.order_id
        )
    DB_PROCESSED_REFUNDS.add(args.idempotency_key)
    return RefundResult(
        status='refund_issued',
        transaction_id='tx_new_8899',
        amount_cents=args.amount_cents,
        order_id=args.order_id
    )

# Tool Capability Registry
TOOL_REGISTRY = {
    'get_order': {'schema': GetOrderArgs, 'func': get_order_impl, 'effect': 'read', 'permission': 'order:read'},
    'issue_refund': {'schema': IssueRefundArgs, 'func': issue_refund_impl, 'effect': 'write', 'permission': 'refund:issue'}
}
print('Tool registry initialized.')

Tool registry initialized.


## Part 6 — Defense-in-Depth Safe Dispatcher
The dispatcher enforces the execution boundary in 5 strict phases:
1. **Tool Existence:** Rejects unapproved/unknown tools.
2. **Actor Authorization:** Verifies actor roles against tool permissions.
3. **Schema Validation:** Strictly validates JSON against Pydantic model (`extra='forbid'`).
4. **Tenant & Business Validation:** Checks organization boundary (`order.tenant_id == ctx.tenant_id`) and domain rules (`amount <= total`).
5. **Execution & Idempotency:** Dispatches to backend handler.

In [6]:
def dispatch_tool(tool_name: str, raw_args: str, ctx: ExecutionContext) -> BaseModel:
    # 1. Tool Existence
    if tool_name not in TOOL_REGISTRY:
        return ErrorResult(error_type='UNKNOWN_TOOL', message=f"Tool '{tool_name}' not found in registry.")
    
    entry = TOOL_REGISTRY[tool_name]
    
    # 2. Actor Role Authorization
    req_perm = entry['permission']
    if req_perm not in ctx.roles:
        return ErrorResult(error_type='AUTH_ERROR', message=f"Permission denied: Actor lacks '{req_perm}' role.")
        
    # 3. Typed Schema Validation
    try:
        validated_args = entry['schema'].model_validate_json(raw_args)
    except ValidationError as e:
        err_msg = ', '.join([f"{err['loc'][0]}: {err['msg']}" for err in e.errors()])
        return ErrorResult(error_type='SCHEMA_ERROR', message=err_msg)

    # 4. Tenant Scope & Business Validation
    order_id = getattr(validated_args, 'order_id', None)
    if order_id:
        if order_id not in DB_ORDERS:
            return ErrorResult(error_type='BUSINESS_ERROR', message=f"Order '{order_id}' not found.")
        order = DB_ORDERS[order_id]
        # Multi-Tenant Isolation Check
        if order['tenant_id'] != ctx.tenant_id:
            return ErrorResult(error_type='AUTH_ERROR', message='Cross-tenant access denied: order belongs to another organization.')
        # Domain Business Rule (Refund <= Order Total)
        if tool_name == 'issue_refund':
            if validated_args.amount_cents > order['total_cents']:
                return ErrorResult(error_type='BUSINESS_ERROR', message=f"Refund amount ({validated_args.amount_cents}¢) exceeds order total ({order['total_cents']}¢).")
            
    # 5. Execution
    try:
        return entry['func'](validated_args)
    except Exception as e:
        return ErrorResult(error_type='BUSINESS_ERROR', message=str(e))

print('Safe dispatcher initialized.')

Safe dispatcher initialized.


## Part 7 — Evaluation Test Suite
We execute test cases covering normal execution, idempotency, cross-tenant isolation, business limit checks, and unauthorized access.

In [7]:
import pandas as pd

tests = [
    {'desc': '1. Success - Valid Refund', 'ctx': ctx_support_agent, 'tool': 'issue_refund', 'args': '{"order_id": "ORD-123", "amount_cents": 1000, "reason": "damaged", "idempotency_key": "key_101"}'},
    {'desc': '2. Idempotency - Duplicate Refund', 'ctx': ctx_support_agent, 'tool': 'issue_refund', 'args': '{"order_id": "ORD-123", "amount_cents": 1000, "reason": "damaged", "idempotency_key": "key_101"}'},
    {'desc': '3. Cross-Tenant Denial (Other Org Order)', 'ctx': ctx_support_agent, 'tool': 'issue_refund', 'args': '{"order_id": "ORD-789", "amount_cents": 1000, "reason": "damaged", "idempotency_key": "key_102"}'},
    {'desc': '4. Business Rule - Refund > Total', 'ctx': ctx_support_agent, 'tool': 'issue_refund', 'args': '{"order_id": "ORD-123", "amount_cents": 99000, "reason": "damaged", "idempotency_key": "key_103"}'},
    {'desc': '5. Unknown Tool Proposal', 'ctx': ctx_support_agent, 'tool': 'drop_table', 'args': '{}'},
    {'desc': '6. Parameter Injection Attempt', 'ctx': ctx_support_agent, 'tool': 'get_order', 'args': '{"order_id": "ORD-123", "tenant_id": "override_admin"}'},
    {'desc': '7. Authorization Denied (Trainee Role)', 'ctx': ctx_read_only_agent, 'tool': 'issue_refund', 'args': '{"order_id": "ORD-123", "amount_cents": 1000, "reason": "damaged", "idempotency_key": "key_104"}'}
]

results = []
print('=== EXECUTING VALIDATION SUITE ===')
for t in tests:
    res = dispatch_tool(t['tool'], t['args'], t['ctx'])
    results.append({
        'Test': t['desc'],
        'Result Type': type(res).__name__,
        'Status / Error': getattr(res, 'status', getattr(res, 'error_type', 'UNKNOWN')),
        'Output': res.model_dump_json()
    })

df_tests = pd.DataFrame(results)
print(df_tests.to_string(index=False))

# Assertions verifying all security & business invariants
assert results[0]['Result Type'] == 'RefundResult' and 'refund_issued' in results[0]['Output']
assert results[1]['Result Type'] == 'RefundResult' and 'already_processed' in results[1]['Output']
assert results[2]['Result Type'] == 'ErrorResult' and 'AUTH_ERROR' in results[2]['Output']
assert results[3]['Result Type'] == 'ErrorResult' and 'BUSINESS_ERROR' in results[3]['Output']
assert results[4]['Result Type'] == 'ErrorResult' and 'UNKNOWN_TOOL' in results[4]['Output']
assert results[5]['Result Type'] == 'ErrorResult' and 'SCHEMA_ERROR' in results[5]['Output']
assert results[6]['Result Type'] == 'ErrorResult' and 'AUTH_ERROR' in results[6]['Output']
print('\nAll 7 boundary validation tests passed successfully!')

=== EXECUTING VALIDATION SUITE ===
                                    Test  Result Type    Status / Error                                                                                                      Output
               1. Success - Valid Refund RefundResult     refund_issued          {"status":"refund_issued","transaction_id":"tx_new_8899","amount_cents":1000,"order_id":"ORD-123"}
       2. Idempotency - Duplicate Refund RefundResult already_processed {"status":"already_processed","transaction_id":"tx_existing_prev","amount_cents":1000,"order_id":"ORD-123"}
3. Cross-Tenant Denial (Other Org Order)  ErrorResult        AUTH_ERROR  {"error_type":"AUTH_ERROR","message":"Cross-tenant access denied: order belongs to another organization."}
       4. Business Rule - Refund > Total  ErrorResult    BUSINESS_ERROR             {"error_type":"BUSINESS_ERROR","message":"Refund amount (99000¢) exceeds order total (5000¢)."}
                5. Unknown Tool Proposal  ErrorResult      UNKNOW

## Part 8 — Bounded Correction Loop
The application loop permits bounded replanning for correctable `SCHEMA_ERROR`s (e.g. omitted required fields). In contrast, `AUTH_ERROR`s are fatal security boundaries and are **never** resent to the model for retries.

In [8]:
def simulate_correction_loop(tool_name: str, raw_args: str, ctx: ExecutionContext):
    max_retries = 2
    for attempt in range(1, max_retries + 1):
        print(f'Attempt {attempt}...')
        res = dispatch_tool(tool_name, raw_args, ctx)
        if isinstance(res, ErrorResult):
            if res.error_type == 'SCHEMA_ERROR':
                print(f'-> Schema error: {res.message}. Model receives feedback and corrects arguments...')
                # Simulated model correction on subsequent turn
                raw_args = '{"order_id": "ORD-123", "amount_cents": 1000, "reason": "damaged", "idempotency_key": "key_corrected"}'
            elif res.error_type == 'AUTH_ERROR':
                print(f'-> Security Authorization Denied: {res.message}. Hard stop!')
                break
            else:
                print(f'-> Business error: {res.message}. Hard stop.')
                break
        else:
            print('-> Success:', res.model_dump_json())
            break

print('--- Recoverable Schema Error ---')
simulate_correction_loop('issue_refund', '{"order_id": "ORD-123"}', ctx_support_agent)
print('\n--- Unrecoverable Auth Error ---')
simulate_correction_loop('issue_refund', '{"order_id": "ORD-123", "amount_cents": 1000, "reason": "damaged", "idempotency_key": "key_x"}', ctx_read_only_agent)

--- Recoverable Schema Error ---
Attempt 1...
-> Schema error: amount_cents: Field required, reason: Field required, idempotency_key: Field required. Model receives feedback and corrects arguments...
Attempt 2...
-> Success: {"status":"refund_issued","transaction_id":"tx_new_8899","amount_cents":1000,"order_id":"ORD-123"}

--- Unrecoverable Auth Error ---
Attempt 1...
-> Security Authorization Denied: Permission denied: Actor lacks 'refund:issue' role.. Hard stop!


## Part 9 — Structured Outputs (Separate from Tool Calling)
Emitting a final structured decision object (e.g. `SupportDecision`) is distinct from calling a backend tool to perform an action.

In [9]:
class SupportDecision(BaseModel):
    category: Literal['refund', 'replacement', 'escalate']
    summary: str
    requires_human_review: bool
    customer_sentiment: Literal['positive', 'neutral', 'negative']

model_final_output = '{"category": "refund", "summary": "Processed $10.00 refund for damaged item.", "requires_human_review": false, "customer_sentiment": "neutral"}'
decision = SupportDecision.model_validate_json(model_final_output)
print('Validated SupportDecision Object:', decision)

Validated SupportDecision Object: category='refund' summary='Processed $10.00 refund for damaged item.' requires_human_review=False customer_sentiment='neutral'


## Part 10 & 11 — Optional Live Model Integration (OpenAI API)
*(Optional)* When `OPENAI_API_KEY` is present, we execute live Tool Calling through our safe dispatcher, and test Real Structured Outputs using the official Pydantic response formatting API.

In [10]:
api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    print('No OPENAI_API_KEY detected in environment. Skipping live OpenAI API calls.')
else:
    from openai import OpenAI
    client = OpenAI(api_key=api_key)

    # 1. Real Tool Calling Hooked to Secure Dispatcher
    print(f'\n--- Live Tool Calling ({MODEL_NAME}) ---')
    openai_tools = [{
        'type': 'function',
        'function': {
            'name': 'get_order',
            'description': 'Look up order details for a specific customer order ID',
            'parameters': GetOrderArgs.model_json_schema()
        }
    }]
    messages = [{'role': 'user', 'content': 'Can you look up customer order ORD-123?'}]

    response = client.chat.completions.create(model=MODEL_NAME, messages=messages, tools=openai_tools)
    msg = response.choices[0].message
    messages.append(msg)

    if msg.tool_calls:
        tc = msg.tool_calls[0].function
        print(f'Model proposed tool: {tc.name} with args: {tc.arguments}')

        # Dispatch strictly through our defense-in-depth dispatcher
        dispatch_result = dispatch_tool(tc.name, tc.arguments, ctx_support_agent)
        print('Dispatcher Output:', dispatch_result.model_dump_json())

        messages.append({
            'role': 'tool',
            'tool_call_id': msg.tool_calls[0].id,
            'name': tc.name,
            'content': dispatch_result.model_dump_json()
        })

        final_res = client.chat.completions.create(model=MODEL_NAME, messages=messages)
        print('\nFinal Model Response:', final_res.choices[0].message.content)

    # 2. Live Structured Output Parsing
    print(f'\n--- Live Structured Outputs ({MODEL_NAME}) ---')
    try:
        parsed_response = client.beta.chat.completions.parse(
            model=MODEL_NAME,
            messages=[{'role': 'user', 'content': 'My order ORD-123 arrived completely shattered. I would like a refund please.'}],
            response_format=SupportDecision
        )
        parsed_decision = parsed_response.choices[0].message.parsed
        print('Parsed SupportDecision Object:', parsed_decision)
    except Exception as e:
        print('Structured output parsing error:', e)

No OPENAI_API_KEY detected in environment. Skipping live OpenAI API calls.
